In [ ]:
# TF-IDF + BiLSTM

import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load data
with open('/kaggle/input/fetched-data-final-2/fetched_data_final_dedup.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
# df = pd.read_json('/kaggle/input/final-bert-bt-dataset/fetched_data_final.json')
df = pd.DataFrame(data)
df['label'] = df['label'].astype(int)

X = df['text']
y = df['label']

df.head()

# TF-IDF vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(X)

# Convert sparse matrix to dense
X_dense = X_tfidf.toarray()

# Normalize (important for neural networks)
scaler = StandardScaler()
X_dense = scaler.fit_transform(X_dense)

# Reshape for BiLSTM: (samples, timesteps, features)
# We treat each TF-IDF feature as a timestep with 1 feature
X_lstm = X_dense.reshape(X_dense.shape[0], X_dense.shape[1], 1)

X_train, X_val, y_train, y_val = train_test_split(
    X_lstm, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class TfidfBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        # Concatenate forward & backward hidden states
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        out = self.fc(h)
        return self.sigmoid(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TfidfBiLSTM(input_dim=1).to(device)
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb).squeeze()
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            preds = model(xb).squeeze().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())

    all_preds = (np.array(all_preds) > 0.5).astype(int)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Loss: {train_loss:.4f} | "
          f"Acc: {acc:.4f} | Prec: {prec:.4f} | "
          f"Rec: {rec:.4f} | F1: {f1:.4f}")
